![image.png](https://i.imgur.com/a3uAqnb.png)

# Preference Optimization with DPO and IPO

- **Dataset**: [UltraFeedback binarized](https://huggingface.co/datasets/trl-lib/ultrafeedback_binarized) from TRL, 62,135 training and 1,000 test pairs of a chosen and a rejected chat response to the same prompt
- **Task**: offline preference optimisation of a chat model from pairwise comparisons
- **Model**: [Qwen/Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) (494M parameters), fine-tuned with a rank-16 LoRA by TRL's `DPOTrainer`, once with the sigmoid (DPO) loss and once with the IPO loss
- **Runtime**: Colab A100 GPU

In this lab, we will:

- load a preference dataset and put it in the explicit prompt, chosen, rejected format the trainer expects
- run the DPO recipe with a LoRA adapter and no second copy of the model, with every configuration knob explained
- switch the loss to IPO with one option and keep everything else fixed
- score held-out pairs by implicit reward accuracy and margin, and by the likelihood ranking that decoding uses
- save the adapter, reload it and check that it reproduces the scores

## What is Direct Preference Optimization?

Direct Preference Optimization (DPO) fits a language model to pairwise preferences without a reward model and without sampling. Under the Bradley-Terry model of preferences and a KL-regularised objective, the optimal reward is a log-ratio between the policy and a frozen reference, so the reward model can be written in terms of the policy itself and trained by logistic regression on the pairs.

$$\mathcal{L}_{\text{DPO}}=-\,\mathbb{E}_{(x,y_w,y_l)\sim\mathcal{D}}\Big[\log\sigma\Big(\beta\log\frac{\pi_\theta(y_w\mid x)}{\pi_{\text{ref}}(y_w\mid x)}-\beta\log\frac{\pi_\theta(y_l\mid x)}{\pi_{\text{ref}}(y_l\mid x)}\Big)\Big]$$

Where:
- $x$ is the prompt, $y_w$ the chosen (winning) response and $y_l$ the rejected (losing) response
- $\pi_\theta$ is the policy being trained and $\pi_{\text{ref}}$ the frozen reference, here the model before training
- $\beta$ sets how far the policy may move from the reference, and $\sigma$ is the logistic function
- $r_\theta(x, y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)}$ is the implicit reward, and the term inside $\sigma$ is the implicit reward margin of the pair

Let $\rho$ be the gap between the two log-ratios. DPO minimises $-\log\sigma(\beta\rho)$, which keeps pushing $\rho$ up for a pair it already ranks correctly. IPO replaces it by the squared loss $(\rho - \tfrac{1}{2\tau})^2$, which regresses the gap onto the finite target $1/(2\tau)$. TRL exposes both as `loss_type`, uses `beta` for $\tau$, and for IPO averages the log-probabilities over the response tokens where DPO sums them.

Papers: [Rafailov et al., 2023. Direct Preference Optimization](https://arxiv.org/abs/2305.18290) and [Azar et al., 2023. A General Theoretical Paradigm to Understand Learning from Human Preferences](https://arxiv.org/abs/2310.12036)

## Setup

In Google Colab, select **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom.

In [ ]:
# trl 0.22.2 predates transformers 5, so the two are pinned together (the same pair as the GRPO lab).
# pip also moves huggingface_hub below 1.0 for transformers 4.57, which datasets, accelerate and peft accept.
%pip install -q trl==0.22.2 transformers==4.57.6
%pip uninstall -q -y torchao

In [ ]:
import gc
import os
import time

import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer, extract_prompt

import torch

Every knob of the recipe lives here. `MAX_STEPS` times `BATCH_SIZE` times `GRAD_ACCUM` equals `N_TRAIN`, so each run is one pass over the training pairs. On a T4 the trainer uses float16 autocast with a gradient scaler, on an A100 bfloat16.

In [ ]:
SEED = 0
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
N_TRAIN = 1600           # training pairs (the split has 62,135)
N_TEST = 200             # held-out pairs from the test split
MAX_CHARS = 1000         # keep pairs whose responses are at most this long, so that nothing is truncated
MAX_LENGTH = 640         # tokens of prompt plus response that the trainer keeps
MAX_PROMPT_LENGTH = 256  # tokens of prompt kept (longer prompts are truncated from the left)
BATCH_SIZE = 4           # pairs per forward pass
GRAD_ACCUM = 2           # forward passes per optimizer step, so 8 pairs per step
MAX_STEPS = 200          # optimizer steps per loss, one pass over N_TRAIN
LR = 5e-5                # LoRA tolerates a higher rate than full fine-tuning (TRL's full fine-tuning example uses 5e-7)
BETA = 0.1               # DPO temperature, and tau for IPO
LORA_RANK = 16
LORA_ALPHA = 32          # adapter scale is LORA_ALPHA / LORA_RANK = 2
LOG_STEPS = 5
NUM_WORKERS = 0          # DataLoader workers, the batches are tiny

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print("No GPU found, so training will be slow. In Colab: Runtime > Change runtime type > T4 GPU.")

# Mixed precision: float16 with a gradient scaler on a T4, which has no bfloat16 units, bfloat16 on Ampere or newer
USE_BF16 = DEVICE == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8
USE_FP16 = DEVICE == "cuda" and not USE_BF16
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

## 1. Dataset

- **Source**: [trl-lib/ultrafeedback_binarized](https://huggingface.co/datasets/trl-lib/ultrafeedback_binarized) on Hugging Face, TRL's binarised version of UltraFeedback (Cui et al., 2023), whose responses were scored by GPT-4
- **Content**: 62,135 training and 1,000 test rows, each with a `chosen` and a `rejected` two-turn conversation (the same user prompt, two assistant answers) and the two judge scores
- **What we use**: 800 training pairs and 200 test pairs whose responses are at most `MAX_CHARS` characters long

The prompt is implicit in this dataset: it is the first turn of both conversations. `extract_prompt` moves it into its own `prompt` column, which is the explicit format the trainer recommends.

In [ ]:
def is_short(example):
    """Keep pairs whose two responses both fit MAX_CHARS, so that MAX_LENGTH truncates nothing."""
    return (len(example["chosen"][-1]["content"]) <= MAX_CHARS
            and len(example["rejected"][-1]["content"]) <= MAX_CHARS)


raw = load_dataset("trl-lib/ultrafeedback_binarized")
train_pairs = (raw["train"].shuffle(seed=SEED).select(range(6 * N_TRAIN))   # a slice, filtered down to N_TRAIN
               .filter(is_short).select(range(N_TRAIN)).map(extract_prompt))
test_short = raw["test"].shuffle(seed=SEED).filter(is_short)
test_pairs = test_short.select(range(min(N_TEST, len(test_short)))).map(extract_prompt)
print(raw)
print(f"{len(train_pairs)} training pairs, {len(test_pairs)} held-out pairs")

One pair, and the judge scores that decided which response is chosen.

In [ ]:
pair = train_pairs[0]
print("prompt  :", pair["prompt"][0]["content"][:300])
print("chosen  :", pair["chosen"][0]["content"][:300])
print("rejected:", pair["rejected"][0]["content"][:300])
print(f"\nscores: chosen {pair['score_chosen']:.1f}, rejected {pair['score_rejected']:.1f}")
gaps = np.array(train_pairs["score_chosen"]) - np.array(train_pairs["score_rejected"])
chosen_len = np.mean([len(c[0]["content"]) for c in train_pairs["chosen"]])
rejected_len = np.mean([len(r[0]["content"]) for r in train_pairs["rejected"]])
print(f"mean score gap {gaps.mean():.2f} | mean length: chosen {chosen_len:.0f}, rejected {rejected_len:.0f} characters")

## 2. Model

The policy is Qwen2.5-0.5B-Instruct in float32 on the GPU, with the matmuls run in float16 by autocast. Training updates a LoRA adapter of rank `LORA_RANK` on every attention and MLP projection, so the base weights stay frozen. That is what lets the trainer skip the second copy of the model: with a PEFT model and no `ref_model`, `DPOTrainer` computes the reference log-probabilities by switching the adapter off, so $\pi_{\text{ref}}$ is the base model and only one set of weights is in memory. We load the policy through a function because the IPO run starts from a fresh copy.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def load_policy():
    """A fresh float32 copy of the instruct model on DEVICE. Autocast runs the matmuls in AMP_DTYPE."""
    return AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(DEVICE)


peft_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

policy = load_policy()
n_total = sum(p.numel() for p in policy.parameters())
print(f"Base parameters: {n_total:,} | vocabulary: {len(tokenizer):,}")

### Scoring held-out pairs

For each held-out pair we compute the log-probability of the chosen and the rejected response under a model, summed over the response tokens, and the number of response tokens. From these numbers under the policy and under the reference we get:

- the implicit reward margin $\beta[(\log\pi_\theta(y_w) - \log\pi_{\text{ref}}(y_w)) - (\log\pi_\theta(y_l) - \log\pi_{\text{ref}}(y_l))]$ and the implicit reward accuracy, the share of pairs with a positive margin, which is what the loss optimises
- the likelihood accuracy, the share of pairs where the policy gives the chosen response the higher log-probability, which is the ranking that decoding uses
- the length-normalised likelihood accuracy, the same with each log-probability divided by its response length

In [ ]:
@torch.no_grad()
def completion_logprob(model, prompt, response):
    """Sum of log p(response tokens | prompt) under model, and the number of response tokens."""
    prompt_ids = tokenizer.apply_chat_template(prompt, add_generation_prompt=True, tokenize=True)
    prompt_ids = list(prompt_ids["input_ids"]) if hasattr(prompt_ids, "keys") else list(prompt_ids)
    response_ids = tokenizer(response[0]["content"] + tokenizer.eos_token, add_special_tokens=False).input_ids
    prompt_ids = prompt_ids[-MAX_PROMPT_LENGTH:]                   # same left truncation as the trainer
    ids = torch.tensor([(prompt_ids + response_ids)[:MAX_LENGTH]], device=DEVICE)        # (1, T)
    with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=DEVICE == "cuda"):
        logits = model(ids).logits[:, :-1]                                                 # (1, T-1, V)
    token_logprobs = torch.log_softmax(logits.float(), dim=-1).gather(-1, ids[:, 1:, None]).squeeze(-1)  # (1, T-1)
    response_logprobs = token_logprobs[0, len(prompt_ids) - 1:]                            # (n_response,)
    return response_logprobs.sum().item(), len(response_logprobs)


def score_pairs(model, dataset, use_adapter=True):
    """Rows of (logp chosen, logp rejected, n chosen, n rejected). use_adapter=False scores the base weights."""
    model.eval()
    rows = []
    for example in dataset:
        if use_adapter or not hasattr(model, "disable_adapter"):
            lp_c, n_c = completion_logprob(model, example["prompt"], example["chosen"])
            lp_r, n_r = completion_logprob(model, example["prompt"], example["rejected"])
        else:
            with model.disable_adapter():
                lp_c, n_c = completion_logprob(model, example["prompt"], example["chosen"])
                lp_r, n_r = completion_logprob(model, example["prompt"], example["rejected"])
        rows.append((lp_c, lp_r, n_c, n_r))
    return np.array(rows)


def metrics(policy_scores, reference_scores, beta):
    """Implicit reward accuracy and margin against the reference, and the two likelihood rankings."""
    lp_c, lp_r, n_c, n_r = policy_scores.T
    ref_c, ref_r = reference_scores[:, 0], reference_scores[:, 1]
    implicit_margin = beta * ((lp_c - ref_c) - (lp_r - ref_r))
    return {"implicit reward acc": float(np.mean(implicit_margin > 0)),
            "implicit margin": float(implicit_margin.mean()),
            "likelihood acc": float(np.mean(lp_c > lp_r)),
            "length-norm acc": float(np.mean(lp_c / np.maximum(n_c, 1) > lp_r / np.maximum(n_r, 1)))}

### Baseline before training

The reference scores come from the base model and serve every later comparison. Scored against itself the reference has a margin of exactly zero on every pair, so its two implicit columns read zero and only its likelihood rankings are informative: how often the instruct model already prefers the chosen response.

In [ ]:
start = time.time()
reference_scores = score_pairs(policy, test_pairs)
scores = {"reference": reference_scores}
results = {"reference": metrics(reference_scores, reference_scores, BETA)}
print({k: round(v, 3) for k, v in results["reference"].items()}, f"| {time.time() - start:.0f}s")

## 3. Training

The configuration, line by line:

- `loss_type`: `"sigmoid"` is the DPO loss $-\log\sigma(\beta\rho)$, `"ipo"` the squared loss $(\rho - 1/2\tau)^2$ on length-averaged log-probabilities. Everything else stays the same between the two runs.
- `beta=BETA`: the temperature on the log-ratios. A larger value keeps the policy closer to the reference. For IPO it is $\tau$, so the target gap is $1/(2\beta) = 5$.
- `learning_rate=LR`: AdamW on the adapter weights, with the trainer's default linear decay and no warm-up.
- `max_steps=MAX_STEPS`, `per_device_train_batch_size=BATCH_SIZE`, `gradient_accumulation_steps=GRAD_ACCUM`: 100 steps of 8 pairs, one pass over `N_TRAIN`.
- `max_length` and `max_prompt_length`: the token budgets. The character filter above keeps every response inside them.
- `fp16` and `bf16`: autocast in the precision the GPU supports. The LoRA weights stay in float32, so the gradient scaler of float16 training has something to scale.
- `gradient_checkpointing=True`: recompute activations in the backward pass. Each pair costs two sequences through the policy and two through the reference.
- `logging_steps=LOG_STEPS`, `report_to="none"`, `save_steps` beyond `MAX_STEPS`: log to the trainer's history and write no checkpoints.
- `peft_config` on the trainer: wrap the policy in the LoRA adapter, and leave `ref_model` unset so that the adapter switched off is the reference.

In [ ]:
def run_dpo(policy, loss_type):
    """Train a fresh LoRA on policy with one DPO loss type and return the trainer."""
    args = DPOConfig(
        loss_type=loss_type,
        beta=BETA,
        learning_rate=LR,
        max_steps=MAX_STEPS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        max_length=MAX_LENGTH,
        max_prompt_length=MAX_PROMPT_LENGTH,
        fp16=USE_FP16,
        bf16=USE_BF16,
        gradient_checkpointing=True,
        logging_steps=LOG_STEPS,
        save_steps=10_000,
        dataloader_num_workers=NUM_WORKERS,
        report_to="none",
        output_dir=f"outputs_{loss_type}",
        seed=SEED,
    )
    trainer = DPOTrainer(model=policy, args=args, train_dataset=train_pairs,
                         processing_class=tokenizer, peft_config=peft_config)
    n_trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    print(f"{loss_type}: {MAX_STEPS} optimizer steps of {BATCH_SIZE * GRAD_ACCUM} pairs, "
          f"{n_trainable:,} trainable LoRA parameters")
    start = time.time()
    trainer.train()
    print(f"{loss_type}: trained in {(time.time() - start) / 60:.1f} minutes")
    return trainer


dpo_trainer = run_dpo(policy, "sigmoid")
dpo_history = dpo_trainer.state.log_history

We score the held-out pairs with the trained adapter, save the adapter and free the GPU before the IPO run.

In [ ]:
scores["DPO"] = score_pairs(dpo_trainer.model, test_pairs)
results["DPO"] = metrics(scores["DPO"], reference_scores, BETA)
print("DPO:", {k: round(v, 3) for k, v in results["DPO"].items()})

dpo_trainer.model.save_pretrained("dpo_sigmoid_lora")           # the adapter alone
del dpo_trainer, policy
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### IPO with one option

A fresh copy of the model, the same data, the same adapter shape, the same steps, and `loss_type="ipo"`.


In [ ]:
policy = load_policy()
ipo_trainer = run_dpo(policy, "ipo")
ipo_history = ipo_trainer.state.log_history

scores["IPO"] = score_pairs(ipo_trainer.model, test_pairs)
results["IPO"] = metrics(scores["IPO"], reference_scores, BETA)
print("IPO:", {k: round(v, 3) for k, v in results["IPO"].items()})

ipo_trainer.model.save_pretrained("dpo_ipo_lora")
del ipo_trainer, policy
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 4. Results

The trainer logs, for every `LOG_STEPS` steps, the implicit reward margin and accuracy on the training batch and the loss. We plot the two runs together.

In [ ]:
def curve(history, key):
    """Steps and values of one logged metric."""
    points = [(e["step"], e[key]) for e in history if key in e]
    return [p[0] for p in points], [p[1] for p in points]


fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name, history in (("DPO", dpo_history), ("IPO", ipo_history)):
    axes[0].plot(*curve(history, "rewards/margins"), marker="o", label=name)
    axes[1].plot(*curve(history, "rewards/accuracies"), marker="o", label=name)
    axes[2].plot(*curve(history, "loss"), marker="o", label=name)
for axis, title in zip(axes, ("Training implicit reward margin", "Training implicit reward accuracy", "Training loss")):
    axis.set(xlabel="step", title=title)
    axis.grid(alpha=0.3)
    axis.legend()
plt.tight_layout()
plt.show()

The held-out table: the base model, the DPO adapter and the IPO adapter on the same pairs.

In [ ]:
columns = list(results["reference"].keys())
print(f"{'held-out pairs':22}" + "".join(f"{c:>22}" for c in columns))
for name, row in results.items():
    print(f"{name:22}" + "".join(f"{row[c]:>22.3f}" for c in columns))

**What to notice**

The training margin grows from zero for both losses and the training accuracy rises above one half. On the held-out pairs the implicit reward accuracy is well above one half for both adapters, while the likelihood accuracy moves much less: the loss ranks pairs by a log-ratio to the reference, decoding ranks them by likelihood, and the two rankings can disagree. The length-normalised ranking is usually closer to the labels than the raw one, because the longer response tends to have the lower total log-probability. IPO's margin levels off near its target where DPO's keeps rising, so compare how far each loss moved as well as its accuracy.

### Save and reload the adapter

`save_pretrained` on the trained model wrote only the adapter (`adapter_config.json` and `adapter_model.safetensors`). We load a fresh base model, attach the DPO adapter with `PeftModel.from_pretrained` and score the first held-out pairs again: the log-probabilities should match the ones computed before the model was freed.

In [ ]:
adapter_bytes = sum(os.path.getsize(os.path.join("dpo_sigmoid_lora", f)) for f in os.listdir("dpo_sigmoid_lora"))
print(f"adapter size on disk: {adapter_bytes / 1e6:.1f} MB")

reloaded = PeftModel.from_pretrained(load_policy(), "dpo_sigmoid_lora")
n_check = 50
reloaded_scores = score_pairs(reloaded, test_pairs.select(range(n_check)))
max_diff = np.abs(reloaded_scores[:, :2] - scores["DPO"][:n_check, :2]).max()
print(f"largest log-probability difference after reload: {max_diff:.4f}")
print("reloaded adapter:", {k: round(v, 3) for k, v in metrics(reloaded_scores, reference_scores[:n_check], BETA).items()})

## Summary

- DPO turns pairwise preferences into a classification loss on the log-ratios between the policy and a frozen reference, with `beta` as the temperature.
- With a LoRA adapter the reference is the same network with the adapter off, so one set of weights serves both roles and a 0.5B model trains on a T4.
- `loss_type` switches between DPO's logistic loss and IPO's squared loss on a finite target, and the rest of the recipe stays fixed.
- The implicit reward accuracy is what the loss optimises, the likelihood ranking is what decoding uses, and the two can disagree.

## Exercises

1. **Temperature.** Set `BETA = 0.5` and retrain with `loss_type="sigmoid"`. Compare the held-out implicit margin and the likelihood accuracy with the run above.
2. **Steps.** Set `MAX_STEPS = 300` and `N_TRAIN = 2400`. Compare the training margin curve and the held-out implicit reward accuracy, and watch the training time.
3. **Learning rate.** Set `LR = 5e-6`, the value of TRL's LoRA example. Compare how far the margin moves in the same number of steps.
4. **Another loss.** Set `loss_type="hinge"` in `run_dpo`, a third member of the same family. Compare its held-out row with DPO and IPO.

## Further Reading

- [Rafailov et al., 2023. Direct Preference Optimization: Your Language Model is Secretly a Reward Model](https://arxiv.org/abs/2305.18290)
- [Azar et al., 2023. A General Theoretical Paradigm to Understand Learning from Human Preferences](https://arxiv.org/abs/2310.12036)
- [Meng et al., 2024. SimPO: Simple Preference Optimization with a Reference-Free Reward](https://arxiv.org/abs/2405.14734)
- [Lambert et al., 2024. Tulu 3: Pushing Frontiers in Open Language Model Post-Training](https://arxiv.org/abs/2411.15124)
- [Cui et al., 2023. UltraFeedback: Boosting Language Models with Scaled AI Feedback](https://arxiv.org/abs/2310.01377)
- [Hu et al., 2021. LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)

### Contributed by: Mohamed Eltayeb